# V5 Feasibility: Hierarchical Stress Decomposition (Iteration 4)

## TL;DR

This diagnostic tests a specific explanation for the V4 symbolic-regression
failure. A single pointwise formula may be trying to learn two different
tasks at once:

1. the overall stress level and upper-tail amplitude of an FEM case; and
2. the normalised spatial stress shape inside that case.

The notebook therefore compares a direct nonlinear control with two
hierarchical reconstructions:

\[
\hat\sigma_{e,c}=\hat B_c+\hat A_c\hat S_{e,c},
\]

where \(B_c\) is case mean stress, \(A_c=\sigma_{95,c}-B_c\), and
\(S_{e,c}\) is the normalised element-level shape.

The **oracle hierarchy** uses the true validation-case \(A_c,B_c\) only
to test shape learnability. It is not deployable. The **deployable
hierarchy** predicts \(A_c,B_c\) exclusively from summaries of available
input fields. The locked 50-case final test is never read.


### Execution note

Run this notebook independently and wait for it to finish before starting another iteration. The models evaluate complete FEM cases and parallel notebook runs would compete for memory. The final-test cases remain sealed.


## Context & Methods

### Key assumptions

- One complete FEM case remains the independent experimental unit.
- The frozen iteration-4 split is used: 119 training, 15 validation
  and 15 internal-test cases. This preserves the pre-registered case-level
  split and similarity-group constraints used throughout CT3.
- All validation and internal-test elements are evaluated. Training uses
  the same deterministic 5,000-row-per-case tail-stratified design already
  audited in V4, so this experiment is directly comparable and quick.
- Exported post-deformation coordinates are treated as available inputs.
- Mean and P95 stress are decomposition targets during training, not model
  inputs. At deployment they are predicted from input-field summaries.
- This is a feasibility diagnostic using nonlinear controls. It does not
  produce or approve the final symbolic formula.

### Fair comparison

The direct and shape models use the same model family, sampled rows,
weights and predictor matrix. The only intentional difference is whether
the target is raw stress or within-case normalised stress. This isolates
the value of the decomposition itself.


## 1. Setup


In [ ]:
from pathlib import Path
import sys
import pandas as pd

CWD = Path.cwd().resolve()
PACKAGE_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'src' / 'ct3_common.py').is_file()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory')

sys.path.insert(0, str(PACKAGE_ROOT / "src"))
from hierarchical_feasibility import FeasibilityConfig, run_feasibility

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

CONFIG = FeasibilityConfig(
    iteration=4,
    rows_per_case=5_000,
    hgb_max_iter=180,
    output_subdir="iteration_4_pilot",
    force_rebuild_sample=False,
)
print("Package root:", PACKAGE_ROOT)
print("Configuration:", CONFIG)


## 2. Run the Diagnostic

The run has four bounded stages: reuse/assemble the audited training
sample, select a case-level amplitude model using validation cases, fit
direct and normalised-shape controls, then evaluate every element in the
validation and internal-test cases.


In [ ]:
RESULTS = run_feasibility(PACKAGE_ROOT, CONFIG)


## 3. Results


In [ ]:
display(RESULTS["case_model_metrics"].sort_values(
    ["split", "selection_score", "mean_p95_relative_error"]
))
display(RESULTS["split_metrics"].sort_values(["split", "model"]))
display(RESULTS["decision_table"])


In [ ]:
from IPython.display import Image, display as show

OUTPUT_DIR = Path(RESULTS["run_summary"]["output_directory"])
for figure_name in [
    "hierarchical_method_comparison.png",
    "validation_p95_adaptation.png",
    "case_amplitude_prediction.png",
]:
    show(Image(filename=str(OUTPUT_DIR / figure_name)))


## Takeaways and Decision Rule

Interpret the three methods separately:

- If the **oracle hierarchy** is clearly better than the direct control,
  normalised shape is learnable and decomposition is structurally useful.
- If oracle is good but the **deployable hierarchy** is poor, the main
  bottleneck is predicting case amplitude/offset from input summaries.
- If both hierarchical methods are comparable to or better than direct on
  validation and internal test, a two-stage symbolic-regression V5 is
  justified.
- If oracle is materially worse than direct, do not spend another long
  PySR run on this hierarchy. Revisit spatial representation or the target
  definition first.

The automated 5% comparison flags are screening aids, not proof. The
final decision must also inspect P95/P99 error, underprediction, hotspot
overlap, case ordering and consistency between validation and internal
test. The final 50 cases remain sealed regardless of this result.
